# 4 · Train the classifier - recognises the top symbol

Trains a small convolutional network (CNN) **from scratch** that gets a crop around one die and says which symbol is on
top (or **“?”** if the die is tilted or not clear). The variants in `TRAIN` are trained (default: only `s025f`, the
fast design - the first stage halves the image right away, about half the compute on the ESP32). To try more designs:
`TRAIN = list(VARIANTS)`. Bigger is usually more accurate, but slower on the ESP32 (column “Compute”). At the end
`CHOSEN` is exported.

**Before:** notebook 2 is done, *Runtime → Change runtime type → L4 GPU*.
**If Colab stops:** run everything again - finished variants are skipped, started ones resume.

## 1 · Settings
The trainer has filled in Drive folder and project. Normally nothing needs to be changed here.

In [ ]:
DRIVE_FOLDER = "__DRIVE__"        # folder in "My Drive"
PROJECT = "__PROJECT__"           # project folder name (under projects/)
NB_NAME, NB_VERSION = "4_classifier", 1   # version of this notebook (to detect updates)
# name: (width, fast[, blur, color_cast]). "fast" = first stage with stride 2: half the compute.
# blur / color_cast (default 1.0) strengthen the degradation during training - for real camera images that are
# blurrier and have a stronger colour cast than the renders.
VARIANTS = {"s025": (0.25, True), "b025": (0.25, False), "b050": (0.5, False), "b100": (1.0, False),
            "s025f": (0.25, True, 1.0, 2.0),     # like s025, double colour cast
            "s025fu": (0.25, True, 1.5, 2.0)}    # like s025, double colour cast + 1.5× blur
EPOCHS = 60
TRAIN = ["s025f"]          # which variants are trained (compare all: list(VARIANTS))
CHOSEN = "s025f"           # exported variant; after the comparison (cell 8) change it if needed, then run cells 1 + 9
MODELS_FOLDER = "models"   # under classifier/; a new name keeps the older models for comparison
COMPARE_ALSO = []          # older models to measure along in cell 8, e.g. ["models_old/s025f"] (under classifier/)
CROPS_FOLDER = "crops"     # under classifier/; crops on Drive (built once, reused in every session)

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
P = f"{DRIVE}/projects/{PROJECT}"
SCRIPTS = f"{DRIVE}/scripts"
SCENE = f"{P}/scene"
RENDERS = f"{P}/renders"
PREVIEW = f"{P}/renders_preview"
EXPORT = f"{P}/export"
DET = f"{P}/detector"
CLS = f"{P}/classifier"
ESP_DETECTION = "/content/esp-detection"
print("Project folder:", P)
CLS_DATA = f"{CLS}/{CROPS_FOLDER}"
MODELS = f"{CLS}/{MODELS_FOLDER}"

## 2 · Check the GPU
Expected: **NVIDIA L4** (faster) or **Tesla T4**. If it says "no GPU": *Runtime → Change runtime type → L4 GPU*
(or T4), then start again from the top.

In [ ]:
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("✔ GPU:", gpu) if gpu else print("✘ no GPU! Runtime → Change runtime type → L4 or T4 GPU, then run everything again.")

## 3 · Connect Google Drive and unpack the newest package
Colab asks whether the notebook may access your Drive → **Allow**.

Then the **newest Colab package** in `packages/` is unpacked (the ZIP file from the trainer - upload it to
*My Drive → __DRIVE__ → packages*, do **not** unpack it). Scripts, notebooks, `project.json`, photos, test images and the
camera setup are updated. **Renders and training results are not touched.**
A package is unpacked only once; a newer package replaces photos, test images and camera setup completely.

In [ ]:
import glob, json, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
PACKAGES = f"{DRIVE}/packages"
os.makedirs(PACKAGES, exist_ok=True)
packages = sorted(glob.glob(f"{PACKAGES}/Colab_{PROJECT}_*.zip"))
state_file = f"{DRIVE}/package_state.json"
state = json.load(open(state_file)) if os.path.exists(state_file) else {}
if not packages:
    print(f"⚠ No package “Colab_{PROJECT}_….zip” in {PACKAGES} - create one in the trainer and upload the ZIP file there.")
elif state.get(PROJECT) == os.path.basename(packages[-1]):
    print("✔ newest package is unpacked already:", os.path.basename(packages[-1]))
else:
    new = packages[-1]
    print("unpacking:", os.path.basename(new))
    with zipfile.ZipFile(new) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
        prefix = names[0].split("/")[0]
        # mirror of the laptop: these folders are replaced completely (deleted photos disappear here too)
        for part in ("scripts", "notebooks", f"projects/{PROJECT}/photos", f"projects/{PROJECT}/test_images",
                     f"projects/{PROJECT}/camera_setup"):
            if any(n.startswith(f"{prefix}/{part}/") for n in names) and os.path.isdir(f"{DRIVE}/{part}"):
                shutil.rmtree(f"{DRIVE}/{part}")
        for n in names:
            target = os.path.join(DRIVE, n.split("/", 1)[1])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(n) as src, open(target, "wb") as f:
                shutil.copyfileobj(src, f)
    state[PROJECT] = os.path.basename(new)
    json.dump(state, open(state_file, "w"), indent=1)
    print(f"✔ unpacked ({len(names)} files)")
    try:
        v = json.load(open(f"{DRIVE}/notebooks/version.json")).get(NB_NAME, 0)
        if v > NB_VERSION:
            print(f"⚠ The package contains a NEWER version of this notebook: please open notebooks/{NB_NAME}.ipynb in Drive again.")
    except (OSError, ValueError):
        pass
missing = [p for p in (f"{P}/project.json", f"{SCRIPTS}/render.py") if not os.path.exists(p)]
if missing:
    print("✘ not found:", *missing, sep="\n   ")
else:
    print("✔ project and scripts found")

## 4 · Install the training programs (about 1-2 minutes, every session)
Fixed versions so that the result stays reproducible: **ultralytics 8.4.189**, **albumentations 2.0.8**,
**onnx 1.23.2**, **onnxsim 0.8.1** and Espressif's **esp-detection** (fixed commit), which describes the detector
*ESPDet-Pico*. PyTorch comes with Colab.

In [ ]:
import os, subprocess, sys, json
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.189", "albumentations==2.0.8",
                "onnx==1.23.2", "onnxsim==0.8.1"], check=True)
if not os.path.isdir(ESP_DETECTION):
    subprocess.run(["git", "clone", "-q", "https://github.com/espressif/esp-detection.git", ESP_DETECTION], check=True)
subprocess.run(["git", "-C", ESP_DETECTION, "checkout", "-q", "28de5902a357d99dd9a5db0f52905255641c5a78"], check=True)
sys.path.insert(0, SCRIPTS)
import torch, ultralytics, ml_tools
d = ml_tools.load_project(P)
IMGSZ, CROP = ml_tools.sizes(d)
print(f"✔ torch {torch.__version__} (CUDA: {torch.cuda.is_available()}), ultralytics {ultralytics.__version__}")
print(f"✔ classes: {', '.join(ml_tools.classes(d))} · detector input {IMGSZ} px · crop {CROP} px")

## 5 · Build the crops (once; depending on Drive 15 minutes to 3 hours)
Cuts every die out of every render (a square around the die with a margin, exactly as later on the ESP32) and stores
them bundled on Drive. Real labelled throws are added as a real test set.
If the crops exist already, this cell takes seconds. New renders are cropped on their own; files whose labels have
changed are detected and rebuilt.

**Duration:** Drive sometimes delivers very slowly (e.g. with two sessions at the same time). The line
“… images/s” shows the speed. This cell needs **no GPU**: if it takes long, stop the cell as soon as a file
`crops_….npz` is finished (it is kept), *Change runtime type → CPU*, run cells 1-5, then L4 again and *Run all*.

In [ ]:
import re, subprocess, time
t0 = time.time()
proc = subprocess.Popen(["python", f"{SCRIPTS}/cls_crops.py", "--project", P, "--renders", RENDERS, "--output", CLS_DATA, "--size", str(CROP)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    z = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", line.rstrip("\n").split("\r")[-1])   # no colour codes
    if z.strip() and (True):
        print(z, flush=True)
proc.wait()
print(f"⏱ {(time.time() - t0) / 60:.1f} min · return code {proc.returncode}")

## 6 · Timing: 1 epoch of the smallest variant

In [ ]:
import time
t0 = time.time()
r = subprocess.run(["python", f"{SCRIPTS}/cls_train.py", "--data", CLS_DATA, "--output", f"{CLS}/timing",
                    "--name", "timing", "--width", "0.25", "--epochs", "1"], capture_output=True, text=True)
s = time.time() - t0
print(r.stdout[-800:] if r.returncode == 0 else r.stdout[-2000:] + r.stderr[-2000:])
print(f"⏱ 1 epoch (width 0.25) incl. start: {s:.0f} s  →  {EPOCHS} epochs about {EPOCHS * s / 60:.0f} min; "
      "width 0.5 / 1.0 take about 1.5× / 3×")

## 7 · Train the variants
Trains every variant in `TRAIN` one after the other. One line per epoch: loss (should go down) and accuracy on the
validation crops (should go up). Finished variants are skipped, started ones resume.

In [ ]:
for name in TRAIN:
    width, fast, *extra = VARIANTS[name]
    blur, color_cast = (extra + [1.0, 1.0])[:2]
    if os.path.exists(f"{MODELS}/{name}.done.json"):
        print(f"✔ {name} is done already"); continue
    cmd = ["python", f"{SCRIPTS}/cls_train.py", "--data", CLS_DATA, "--output", MODELS, "--name", name,
           "--width", str(width), "--epochs", str(EPOCHS)] + (["--fast"] if fast else []) + \
          (["--blur", str(blur)] if blur != 1.0 else []) + (["--color-cast", str(color_cast)] if color_cast != 1.0 else [])
    if width >= 1.0:
        cmd += ["--lr", "0.001"]   # large variants need a smaller learning rate
    if os.path.exists(f"{MODELS}/{name}.ckpt"):
        cmd.append("--resume")
    print(f"=== {name} (width {width}{', fast' if fast else ''}"
          f"{f', blur {blur}' if blur != 1.0 else ''}{f', colour cast {color_cast}' if color_cast != 1.0 else ''}) ===")
    import re, subprocess, time
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        z = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", line.rstrip("\n").split("\r")[-1])   # no colour codes
        if z.strip() and (True):
            print(z, flush=True)
    proc.wait()
    print(f"⏱ {(time.time() - t0) / 60:.1f} min · return code {proc.returncode}")

## 8 · Comparison
Table with size (parameters, estimated size on the ESP32), accuracy **per die** and **per throw** (all 5 dice right)
and the most frequent confusions. The values on **real** images matter most. It is measured **every time anew**
(test only, no new training), so real images you added later (new package, cell 5) are included right away.
Rule of thumb: take the variant with the smallest compute that is hardly worse per throw than the best one.

In [ ]:
from IPython.display import Markdown, display
done = [n for n in VARIANTS if os.path.exists(f"{MODELS}/{n}.done.json")]
also = [f"{CLS}/{a}.pth" for a in COMPARE_ALSO if os.path.exists(f"{CLS}/{a}.pth")]
r = subprocess.run(["python", f"{SCRIPTS}/cls_train.py", "--data", CLS_DATA, "--output", MODELS,
                    "--compare", *done, *also], capture_output=True, text=True)
print(r.stderr[-1500:] if r.returncode else "")
display(Markdown(open(f"{MODELS}/compare.md", encoding="utf-8").read()))

## 9 · Prepare the chosen variant for the ESP32 (export)
Saves `export/dice_cls.onnx` with calibration and test crops (`cls_calib.npz`, `cls_test.npz`, `cls_info.json`).
Afterwards it continues **on the laptop**.

In [ ]:
import re, subprocess, time
t0 = time.time()
proc = subprocess.Popen(["python", f"{SCRIPTS}/cls_export.py", "--model", f"{MODELS}/{CHOSEN}.pth", "--data", CLS_DATA, "--project", P, "--output", EXPORT], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    z = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", line.rstrip("\n").split("\r")[-1])   # no colour codes
    if z.strip() and (True):
        print(z, flush=True)
proc.wait()
print(f"⏱ {(time.time() - t0) / 60:.1f} min · return code {proc.returncode}")
for f in sorted(os.listdir(EXPORT)):
    print(f"  {f:24s} {os.path.getsize(os.path.join(EXPORT, f)) / 1024:8.1f} KB")

## Continue on the laptop
Drive now holds `projects/__PROJECT__/export/` with both models. Download the folder and put it into the project folder
of the trainer (`export/`). The commands for converting, checking and flashing are in the trainer on the page
**“Training & device”**.

## End the session (optional)
A connected Colab session uses **compute units even when it does nothing**. If you leave while something is still
running: set `END_SESSION = True` below and start this cell with ▶ - it queues up and ends the session when all cells
before it are done. All results are on Drive, nothing is lost.
**Note:** if a cell before it stops with an error, Colab discards the waiting cells - this one too. The session then
keeps running until Colab disconnects it (or you choose *Runtime → Disconnect and delete runtime*).
With *Run all* nothing happens here as long as `END_SESSION = False`.

In [ ]:
END_SESSION = False   # <- set to True and press ▶ if the session should end after the running cells
if END_SESSION:
    from google.colab import runtime
    print("Ending the session - results are on Drive.")
    runtime.unassign()
else:
    print("Session keeps running (END_SESSION = False).")